# Compile robustness results: original + 2 new seeds -> mean / std dev

Run `robustness_pipeline.ipynb`/`robustness_pmsd.ipynb`/`robustness_baseline.ipynb`/`robustness_ts_models.ipynb`/`robustness_prophet.ipynb` first -- this only reads their output. `n_runs` < 3 means a combination hasn't fully finished yet.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent.parent
RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
NEW_SEEDS = [43, 44]
SERIES = ["concurrent_cases", "throughput_time"]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets")

In [ ]:
def _original_ppm_metrics(approach: str, dataset: str, sub: str, regime: str = "full") -> pd.DataFrame:
    """approach in {'camargo', 'bukhsh', 'amiri'}. sub in {'synthetic', 'ssd'}.
    regime in {'full', 'half', 'pf'}. bukhsh returns two rows per series,
    tagged 'bukhsh_suffix'/'bukhsh_rt'."""
    trim = "none" if sub == "synthetic" else "ssd"
    run = f"{dataset}_test_full"

    if regime == "full":
        base = RESULTS / f"{approach}_hpo" / trim / run
    elif regime == "half":
        base = RESULTS / f"{approach}_hpo_half" / trim / run
    else:
        base = None

    if approach == "bukhsh":
        frames = []
        if regime == "pf":
            candidates = [
                (RESULTS / "plain_field" / "bukhsh_suffix" / trim / run / f"metrics_{run}.csv", "bukhsh_suffix"),
                (RESULTS / "plain_field" / "bukhsh_rt" / trim / run / f"metrics_{run}.csv", "bukhsh_rt"),
            ]
        else:
            candidates = [
                (base / f"metrics_{run}.csv", "bukhsh_suffix"),
                (base / f"metrics_{run}_rt.csv", "bukhsh_rt"),
            ]
        for p, model_label in candidates:
            if p.exists():
                d = pd.read_csv(p)[["series", "mae", "mse"]].copy()
                d["model"] = model_label
                frames.append(d)
        if not frames:
            return pd.DataFrame()
        df = pd.concat(frames, ignore_index=True)
    else:
        p = (RESULTS / "plain_field" / approach / trim / run / f"metrics_{run}.csv") if regime == "pf" \
            else (base / f"metrics_{run}.csv")
        if not p.exists():
            return pd.DataFrame()
        df = pd.read_csv(p)[["series", "mae", "mse"]].copy()
        df["model"] = approach

    df["dataset"] = dataset
    df["seed"] = "original"
    df["regime"] = regime
    return df


def _original_ts_metrics(dataset: str, sub: str, models: list[str]) -> pd.DataFrame:
    """Reads metrics_<ds>.csv (or, ssd only, a fallback bare metrics.csv --
    real-life's batch writes that name instead), filtered to the requested
    model(s). val_mean is never in this file -- see _original_val_mean_metrics."""
    d = (RESULTS / "ssd" / dataset) if sub == "ssd" else (RESULTS / "synthetic" / "none" / dataset)
    p = d / f"metrics_{dataset}.csv"
    if not p.exists():
        if sub != "ssd" or not (d / "metrics.csv").exists():
            return pd.DataFrame()
        p = d / "metrics.csv"
    df = pd.read_csv(p)
    df = df[df["model"].isin(models)][["series", "model", "mae", "mse"]].copy()
    df["dataset"] = dataset
    df["seed"] = "original"
    df["regime"] = "full"
    return df


def _original_val_mean_metrics(dataset: str, sub: str) -> pd.DataFrame:
    """val_mean's 'original' metrics live in their own dedicated
    metrics_<ds>_val_mean.csv, never the shared metrics_<ds>.csv."""
    d = (RESULTS / "ssd" / dataset) if sub == "ssd" else (RESULTS / "synthetic" / "none" / dataset)
    p = d / f"metrics_{dataset}_val_mean.csv"
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df[["series", "mae", "mse"]].copy()
    df["model"] = "val_mean"
    df["dataset"] = dataset
    df["seed"] = "original"
    df["regime"] = "full"
    return df


def _original_prophet_metrics(dataset: str, sub: str) -> pd.DataFrame:
    """Prophet's 'original' lives in the shared metrics_<ds>.csv, but gets
    its own approach='prophet' tag (see robustness_prophet.ipynb)."""
    return _original_ts_metrics(dataset, sub, ["prophet"])


def _original_pmsd_metrics(dataset: str, sub: str) -> pd.DataFrame:
    d = (RESULTS / "ssd" / dataset) if sub == "ssd" else (RESULTS / "synthetic" / "none" / dataset)
    p = d / f"metrics_{dataset}_pmsd.csv"
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df = df[["series", "mae", "mse"]].copy()
    df["model"] = "pmsd"
    df["dataset"] = dataset
    df["seed"] = "original"
    df["regime"] = "full"
    return df


def _robustness_seed_metrics(approach: str, dataset: str, sub: str, regime: str = "full",
                             exclude_models: tuple[str, ...] = ()) -> pd.DataFrame:
    """Reads robustness/<approach>/<sub>/<dataset>/seed_<N>/metrics.csv
    (or its half/pf subdirectory) for each seed in NEW_SEEDS.

    exclude_models: robustness_ts_models.ipynb's own internal sweep also
    computes 'naive'/'prophet' for internal reference -- exclude them when
    reading the 'ts_models' approach so they don't leak in as phantom
    duplicates alongside their own dedicated approach tags."""
    rel = "metrics.csv" if regime == "full" else f"{regime}/metrics.csv"
    rows = []
    for seed in NEW_SEEDS:
        p = ROBUST / approach / sub / dataset / f"seed_{seed}" / rel
        if not p.exists():
            continue
        df = pd.read_csv(p)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    if exclude_models:
        out = out[~out["model"].isin(exclude_models)]
    out["regime"] = regime
    return out


In [ ]:
def build_robustness_table(sub: str) -> pd.DataFrame:
    """sub in {'synthetic', 'ssd'}. Combines each approach's original run
    with its new-seed robustness runs, computing mean/std across all runs
    per (approach, model, regime, dataset, series)."""
    datasets = SYNTH_DATASETS if sub == "synthetic" else REAL_DATASETS
    all_rows = []

    for dataset in datasets:
        for approach in ("camargo", "bukhsh", "amiri"):
            for regime in ("full", "half", "pf"):
                orig = _original_ppm_metrics(approach, dataset, sub, regime)
                if not orig.empty:
                    orig["approach"] = approach
                    all_rows.append(orig)
                seeds = _robustness_seed_metrics(approach, dataset, sub, regime)
                if not seeds.empty:
                    seeds["approach"] = approach
                    all_rows.append(seeds)

        orig_naive = _original_ts_metrics(dataset, sub, ["naive"])
        if not orig_naive.empty:
            orig_naive["approach"] = "baseline"
            all_rows.append(orig_naive)
        orig_val_mean = _original_val_mean_metrics(dataset, sub)
        if not orig_val_mean.empty:
            orig_val_mean["approach"] = "baseline"
            all_rows.append(orig_val_mean)
        seeds_baseline = _robustness_seed_metrics("baseline", dataset, sub)
        if not seeds_baseline.empty:
            seeds_baseline["approach"] = "baseline"
            all_rows.append(seeds_baseline)

        orig_prophet = _original_prophet_metrics(dataset, sub)
        if not orig_prophet.empty:
            orig_prophet["approach"] = "prophet"
            all_rows.append(orig_prophet)
        seeds_prophet = _robustness_seed_metrics("prophet", dataset, sub)
        if not seeds_prophet.empty:
            seeds_prophet["approach"] = "prophet"
            all_rows.append(seeds_prophet)

        for model in ["ets", "sarimax", "theta", "stl",
                     "ridge", "ridge_mimo", "gru", "gru_mimo", "nbeats", "tft"]:
            orig = _original_ts_metrics(dataset, sub, [model])
            if not orig.empty:
                orig["approach"] = "ts_models"
                all_rows.append(orig)
        seeds_ts = _robustness_seed_metrics("ts_models", dataset, sub, exclude_models=("naive", "prophet"))
        if not seeds_ts.empty:
            seeds_ts["approach"] = "ts_models"
            all_rows.append(seeds_ts)

        orig_pmsd = _original_pmsd_metrics(dataset, sub)
        if not orig_pmsd.empty:
            orig_pmsd["approach"] = "pmsd"
            all_rows.append(orig_pmsd)
        seeds_pmsd = _robustness_seed_metrics("pmsd", dataset, sub)
        if not seeds_pmsd.empty:
            seeds_pmsd["approach"] = "pmsd"
            all_rows.append(seeds_pmsd)

    if not all_rows:
        return pd.DataFrame()
    long_df = pd.concat(all_rows, ignore_index=True)

    summary = (
        long_df.groupby(["approach", "model", "regime", "dataset", "series"])
        .agg(n_runs=("mae", "size"), mae_mean=("mae", "mean"), mae_std=("mae", "std"),
            mse_mean=("mse", "mean"), mse_std=("mse", "std"))
        .reset_index()
    )
    return summary, long_df


## Part 1: Synthetic

In [ ]:
robustness_synth_summary, robustness_synth_long = build_robustness_table("synthetic")
robustness_synth_summary.to_csv(ROBUST / "results" / "robustness_summary_synthetic.csv", index=False)
robustness_synth_long.to_csv(ROBUST / "results" / "robustness_raw_synthetic.csv", index=False)
print(f"{len(robustness_synth_summary)} (approach, model, dataset, series) rows, "
     f"{int((robustness_synth_summary['n_runs'] < 3).sum())} with fewer than 3 runs available so far")
robustness_synth_summary

## Part 2: SSD

In [ ]:
robustness_ssd_summary, robustness_ssd_long = build_robustness_table("ssd")
robustness_ssd_summary.to_csv(ROBUST / "results" / "robustness_summary_ssd.csv", index=False)
robustness_ssd_long.to_csv(ROBUST / "results" / "robustness_raw_ssd.csv", index=False)
print(f"{len(robustness_ssd_summary)} (approach, model, dataset, series) rows, "
     f"{int((robustness_ssd_summary['n_runs'] < 3).sum())} with fewer than 3 runs available so far")
robustness_ssd_summary

## Part 3: Coefficient of variation (cv_pct = 100 * mae_std / mae_mean) — CC and TT separately, transposed

In [ ]:
import numpy as np

_combined = pd.concat([
    robustness_synth_summary.assign(sub="synthetic"),
    robustness_ssd_summary.assign(sub="ssd"),
], ignore_index=True)

_combined["cv_pct"] = np.where(
    (_combined["n_runs"] >= 2) & (_combined["mae_mean"] != 0),
    100 * _combined["mae_std"] / _combined["mae_mean"],
    np.nan,
)

_approach_order = ["camargo", "bukhsh", "amiri", "ts_models", "prophet", "baseline", "pmsd"]
_regime_order = ["full", "half", "pf"]
_row_key = lambda c: (
    _approach_order.index(c[0]) if c[0] in _approach_order else len(_approach_order),
    c[1], _regime_order.index(c[2]) if c[2] in _regime_order else len(_regime_order),
)
_col_key = lambda c: (0 if c[0] == "ssd" else 1, c[1])


def _cv_pct_table(series_name: str) -> pd.DataFrame:
    sub = _combined[_combined["series"] == series_name]
    table = sub.pivot_table(index=["approach", "model", "regime"], columns=["sub", "dataset"], values="cv_pct").round(1)
    table = table.reindex(sorted(table.index, key=_row_key))
    table = table[sorted(table.columns, key=_col_key)]
    return table


robustness_cv_pct_cc = _cv_pct_table("concurrent_cases")
robustness_cv_pct_tt = _cv_pct_table("throughput_time")

robustness_cv_pct_cc.to_csv(ROBUST / "results" / "robustness_cv_pct_cc.csv")
robustness_cv_pct_tt.to_csv(ROBUST / "results" / "robustness_cv_pct_tt.csv")
print(f"CC: {robustness_cv_pct_cc.shape[0]} (approach, model, regime) rows x {robustness_cv_pct_cc.shape[1]} logs")
print(f"TT: {robustness_cv_pct_tt.shape[0]} (approach, model, regime) rows x {robustness_cv_pct_tt.shape[1]} logs")
print(f"saved -> {ROBUST / 'results' / 'robustness_cv_pct_cc.csv'}")
print(f"saved -> {ROBUST / 'results' / 'robustness_cv_pct_tt.csv'}")

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 240)


In [ ]:
print("=== Concurrent Cases: CV% ===")
robustness_cv_pct_cc

In [ ]:
print("=== Avg Throughput Time: CV% ===")
robustness_cv_pct_tt

## Part 4: Best (minimum) MAE across the 3 runs — synthetic and ssd, CC and TT separately

In [ ]:
def _best_mae_table(long_df: pd.DataFrame, series_name: str) -> pd.DataFrame:
    """Returns a (approach, model, regime) x dataset table of the minimum mae across all runs."""
    sub = long_df[long_df["series"] == series_name]
    table = sub.pivot_table(index=["approach", "model", "regime"], columns="dataset",
                            values="mae", aggfunc="min").round(1)
    table = table.reindex(sorted(table.index, key=_row_key))
    table = table[sorted(table.columns)]
    return table


robustness_best_mae_synthetic_cc = _best_mae_table(robustness_synth_long, "concurrent_cases")
robustness_best_mae_synthetic_tt = _best_mae_table(robustness_synth_long, "throughput_time")
robustness_best_mae_ssd_cc = _best_mae_table(robustness_ssd_long, "concurrent_cases")
robustness_best_mae_ssd_tt = _best_mae_table(robustness_ssd_long, "throughput_time")

for _name, _tbl in [
    ("robustness_best_mae_synthetic_cc", robustness_best_mae_synthetic_cc),
    ("robustness_best_mae_synthetic_tt", robustness_best_mae_synthetic_tt),
    ("robustness_best_mae_ssd_cc", robustness_best_mae_ssd_cc),
    ("robustness_best_mae_ssd_tt", robustness_best_mae_ssd_tt),
]:
    _tbl.to_csv(ROBUST / "results" / f"{_name}.csv")
    print(f"{_name}: {_tbl.shape[0]} rows x {_tbl.shape[1]} datasets -> "
          f"{ROBUST / 'results' / (_name + '.csv')}")

In [ ]:
print("=== Synthetic -- Concurrent Cases: best MAE across 3 runs ===")
robustness_best_mae_synthetic_cc

In [ ]:
print("=== Synthetic -- Avg Throughput Time: best MAE across 3 runs ===")
robustness_best_mae_synthetic_tt

In [ ]:
print("=== SSD (real-life) -- Concurrent Cases: best MAE across 3 runs ===")
robustness_best_mae_ssd_cc

In [ ]:
print("=== SSD (real-life) -- Avg Throughput Time: best MAE across 3 runs ===")
robustness_best_mae_ssd_tt

## Part 5: Average MAE across all logs (mean ± sd), saved to Excel

In [ ]:
def average_across_logs(summary_df: pd.DataFrame) -> pd.DataFrame:
    """(approach, model, regime) -> mean +/- sd of mae_mean across every log in summary_df."""
    rows = []
    for (approach, model, regime), grp in summary_df.groupby(["approach", "model", "regime"]):
        row = {"approach": approach, "model": model, "regime": regime}
        for series_name, label in [("concurrent_cases", "cc"), ("throughput_time", "tt")]:
            vals = grp.loc[grp["series"] == series_name, "mae_mean"].dropna()
            row[f"{label}_n_logs"] = len(vals)
            row[f"{label}_mae_mean"] = vals.mean() if len(vals) else None
            row[f"{label}_mae_sd"] = vals.std() if len(vals) > 1 else (0.0 if len(vals) == 1 else None)
        rows.append(row)
    out = pd.DataFrame(rows)
    out = out.set_index(["approach", "model", "regime"])
    out = out.reindex(sorted(out.index, key=_row_key)).reset_index()
    return out


def _fmt_mean_sd(df: pd.DataFrame, label: str) -> pd.Series:
    mean, sd = df[f"{label}_mae_mean"], df[f"{label}_mae_sd"]
    return [f"{m:.2f} \u00b1 {s:.2f}" if pd.notna(m) else "" for m, s in zip(mean, sd)]


robustness_avg_synth = average_across_logs(robustness_synth_summary)
robustness_avg_ssd = average_across_logs(robustness_ssd_summary)

robustness_avg_synth.to_csv(ROBUST / "results" / "robustness_avg_across_logs_synthetic.csv", index=False)
robustness_avg_ssd.to_csv(ROBUST / "results" / "robustness_avg_across_logs_ssd.csv", index=False)

_xlsx_path = ROBUST / "results" / "robustness_avg_across_logs.xlsx"
with pd.ExcelWriter(_xlsx_path, engine="openpyxl") as _writer:
    for _sheet_name, _df in [("Synthetic", robustness_avg_synth), ("SSD", robustness_avg_ssd)]:
        _display = pd.DataFrame({
            "approach": _df["approach"], "model": _df["model"], "regime": _df["regime"],
            "cc_mae (mean \u00b1 sd)": _fmt_mean_sd(_df, "cc"), "cc_n_logs": _df["cc_n_logs"],
            "tt_mae (mean \u00b1 sd)": _fmt_mean_sd(_df, "tt"), "tt_n_logs": _df["tt_n_logs"],
        })
        _display.to_excel(_writer, sheet_name=_sheet_name, index=False)
        _ws = _writer.sheets[_sheet_name]
        for _col_cells in _ws.columns:
            _width = max(len(str(_c.value)) if _c.value is not None else 0 for _c in _col_cells) + 2
            _ws.column_dimensions[_col_cells[0].column_letter].width = _width

print(f"saved -> {_xlsx_path}")
robustness_avg_synth


In [ ]:
DISPLAY_NAME = {
    ("baseline", "naive", "full"): "naive",
    ("baseline", "seasonal_naive", "full"): "seasonal_naive",
    ("baseline", "val_mean", "full"): "val_average",

    ("pmsd", "pmsd", "full"): "PMSD",
    ("camargo", "camargo", "full"): "GLSTM full",
    ("camargo", "camargo", "half"): "GLSTM half prefix",
    ("camargo", "camargo", "pf"): "PF Camargo",

    ("amiri", "amiri", "full"): "Amiri full",
    ("amiri", "amiri", "half"): "Amiri half",
    ("amiri", "amiri", "pf"): "PF Amiri",

    ("bukhsh", "bukhsh_rt", "full"): "PT RT full",
    ("bukhsh", "bukhsh_rt", "half"): "PT RT half prefix",
    ("bukhsh", "bukhsh_rt", "pf"): "PF Bukhsh RT",

    ("bukhsh", "bukhsh_suffix", "full"): "PT suffix full",
    ("bukhsh", "bukhsh_suffix", "half"): "PT suffix half prefix",
    ("bukhsh", "bukhsh_suffix", "pf"): "PF Bukhsh suffix",

    ("prophet", "prophet", "full"): "prophet",

    ("ts_models", "ets", "full"): "ets",
    ("ts_models", "sarimax", "full"): "sarimax",
    ("ts_models", "theta", "full"): "theta",
    ("ts_models", "stl", "full"): "stl",

    ("ts_models", "ridge", "full"): "ridge",
    ("ts_models", "ridge_mimo", "full"): "ridge_mimo",

    ("ts_models", "gru", "full"): "gru",
    ("ts_models", "gru_mimo", "full"): "gru_mimo",

    ("ts_models", "nbeats", "full"): "nbeats",
    ("ts_models", "nhits", "full"): "nhits",
    ("ts_models", "tft", "full"): "tft",

    ("tabpfn", "tabpfn", "full"): "tabpfn",

}

MODEL_ORDER = [
    "naive",
    "seasonal_naive",
    "val_average",
    "ets",
    "sarimax",
    "theta",
    "stl",
    "ridge",
    "ridge_mimo",
    "gru",
    "gru_mimo",
    "nbeats",
    "nhits",
    "tft",
    "prophet",
    "chronos",
    "tabpfn",
    "PMSD",
    "Simod",
    "AgentSimulator",
    "PF Amiri",
    "PF Camargo",
    "PF Bukhsh RT",
    "PF Bukhsh suffix",
    "GLSTM full",
    "Amiri full",
    "PT suffix full",
    "PT RT full",
    "GLSTM half prefix",
    "Amiri half",
    "PT suffix half prefix",
    "PT RT half prefix",
]
def create_result_table(df: pd.DataFrame, series: str) -> pd.DataFrame:
    tmp = df[df["series"] == series].copy()

    tmp["display_model"] = tmp.apply(
        lambda r: DISPLAY_NAME.get(
            (r["approach"], r["model"], r["regime"])
        ),
        axis=1,
    )



    tmp["value"] = (
        tmp["mae_mean"].map(lambda x: f"{x:.2f}")
        + " ± "
        + tmp["mae_std"].map(lambda x: f"{x:.2f}")
    )

    out = tmp.pivot(
        index="display_model",
        columns="dataset",
        values="value",
    )

    out = out.reindex(MODEL_ORDER)

    return out.reset_index().rename(
        columns={"display_model": "model"}
    )
synth_cc = create_result_table(
    robustness_synth_summary,
    "concurrent_cases"
)

synth_tt = create_result_table(
    robustness_synth_summary,
    "throughput_time"
)

ssd_cc = create_result_table(
    robustness_ssd_summary,
    "concurrent_cases"
)

ssd_tt = create_result_table(
    robustness_ssd_summary,
    "throughput_time"
)

with pd.ExcelWriter(
    ROBUST / "results" / "robustness_by_dataset.xlsx",
    engine="openpyxl"
) as writer:

    synth_cc.to_excel(writer, sheet_name="Synthetic_CC", index=False)
    synth_tt.to_excel(writer, sheet_name="Synthetic_TT", index=False)
    ssd_cc.to_excel(writer, sheet_name="SSD_CC", index=False)
    ssd_tt.to_excel(writer, sheet_name="SSD_TT", index=False)